In [1]:
import csv
import shutil
import os
import cv2
import numpy as np
import random
import matplotlib.pyplot as plt
from pathlib import Path

In [2]:
ROOT = "/kaggle/input/smoke-fire-detection-yolo/data"
BAD_IMAGES_DIR = "/kaggle/working/bad_images"
BAD_LABELS_DIR = "/kaggle/working/bad_labels"
CLEAN_ROOT = "/kaggle/working/clean_data"
REPORT_CSV = "/kaggle/working/data_clean_report.csv"

In [3]:
bad_list = [
    "/kaggle/input/smoke-fire-detection-yolo/data/train/images/PublicDataset01005.jpg",
    "/kaggle/input/smoke-fire-detection-yolo/data/train/images/PublicDataset00864.jpg",
    "/kaggle/input/smoke-fire-detection-yolo/data/train/images/WEB04185.jpg",
    "/kaggle/input/smoke-fire-detection-yolo/data/train/images/WEB02767.jpg",
    "/kaggle/input/smoke-fire-detection-yolo/data/train/images/PublicDataset01036.jpg",
    "/kaggle/input/smoke-fire-detection-yolo/data/train/images/PublicDataset00934.jpg",
    "/kaggle/input/smoke-fire-detection-yolo/data/train/images/WEB06024.jpg",
    "/kaggle/input/smoke-fire-detection-yolo/data/train/images/WEB02729.jpg",
    "/kaggle/input/smoke-fire-detection-yolo/data/train/images/WEB06022.jpg",
    "/kaggle/input/smoke-fire-detection-yolo/data/train/images/WEB06334.jpg",
    "/kaggle/input/smoke-fire-detection-yolo/data/train/images/WEB04554.jpg",
    "/kaggle/input/smoke-fire-detection-yolo/data/train/images/WEB06336.jpg",
    "/kaggle/input/smoke-fire-detection-yolo/data/train/images/WEB03587.jpg",
    "/kaggle/input/smoke-fire-detection-yolo/data/train/images/WEB06082.jpg",
    "/kaggle/input/smoke-fire-detection-yolo/data/train/images/PublicDataset00900.jpg",
    "/kaggle/input/smoke-fire-detection-yolo/data/val/images/WEB02650.jpg",
    "/kaggle/input/smoke-fire-detection-yolo/data/val/images/WEB02614.jpg",
    "/kaggle/input/smoke-fire-detection-yolo/data/val/images/PublicDataset01007.jpg",
    "/kaggle/input/smoke-fire-detection-yolo/data/test/images/WEB10194.jpg",
    "/kaggle/input/smoke-fire-detection-yolo/data/test/images/WEB10116.jpg",
    "/kaggle/input/smoke-fire-detection-yolo/data/test/images/WEB10972.jpg",
    "/kaggle/input/smoke-fire-detection-yolo/data/test/images/WEB10874.jpg",
]

In [ ]:
os.makedirs(BAD_IMAGES_DIR, exist_ok=True)
os.makedirs(BAD_LABELS_DIR, exist_ok=True)

for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(CLEAN_ROOT, split, "images"), exist_ok=True)
    os.makedirs(os.path.join(CLEAN_ROOT, split, "labels"), exist_ok=True)

report_rows = []

for split in ["train", "val", "test"]:
    src_img_dir = os.path.join(ROOT, split, "images")
    src_lbl_dir = os.path.join(ROOT, split, "labels")
    dst_img_dir = os.path.join(CLEAN_ROOT, split, "images")
    dst_lbl_dir = os.path.join(CLEAN_ROOT, split, "labels")

    if not os.path.isdir(src_img_dir):
        print("Missing dir:", src_img_dir)
        continue

    for fname in sorted(os.listdir(src_img_dir)):
        src_img = os.path.join(src_img_dir, fname)

        if not os.path.isfile(src_img):
            continue

        name, _ = os.path.splitext(fname)
        src_label = os.path.join(src_lbl_dir, name + ".txt")

        if src_img in bad_list:
            shutil.copy2(src_img, BAD_IMAGES_DIR)

            if os.path.exists(src_label):
                shutil.copy2(src_label, BAD_LABELS_DIR)
                report_rows.append([split, fname, "bad_image_and_label_copied"])
            else:
                report_rows.append([split, fname, "bad_image_copied_label_missing"])
            continue

        img = cv2.imread(src_img)

        if img is None or img.size == 0:
            print(f"Invalid image: {src_img}")
            shutil.copy2(src_img, BAD_IMAGES_DIR)

            if os.path.exists(src_label):
                shutil.copy2(src_label, BAD_LABELS_DIR)
                report_rows.append([split, fname, "corrupted_image_and_label_copied"])
            else:
                report_rows.append([split, fname, "corrupted_image_copied_label_missing"])
            continue

        shutil.copy2(src_img, dst_img_dir)

        if os.path.exists(src_label):
            shutil.copy2(src_label, dst_lbl_dir)
            report_rows.append([split, fname, "copied_with_label"])
        else:
            report_rows.append([split, fname, "copied_label_missing"])

print("=" * 50)

for split in ["train", "val", "test"]:
    clean_dir = os.path.join(CLEAN_ROOT, split, "images")
    if os.path.exists(clean_dir):
        print(f"{split}: {len(os.listdir(clean_dir))} Valid Images")

In [ ]:
print("=" * 50)
print(f"   Invlaid Images Save Her {BAD_IMAGES_DIR}")
print(f" Invlaid Labels Save Her: {BAD_LABELS_DIR}")

with open(REPORT_CSV, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["split", "filename", "action"])
    w.writerows(report_rows)

In [ ]:
yaml_text = f"""train: {CLEAN_ROOT}/train/images
val: {CLEAN_ROOT}/val/images
test: {CLEAN_ROOT}/test/images

nc: 2
names: ['smoke','fire']
"""
with open("/kaggle/working/data.yaml", "w", encoding="utf-8") as yf:
    yf.write(yaml_text)

print("dat.yaml is Done Created")

In [ ]:


train_img_path = "/kaggle/input/smoke-fire-detection-yolo/data/train/images"
train_label_path = "/kaggle/input/smoke-fire-detection-yolo/data/train/labels"

sample_imgs = random.sample(os.listdir(train_img_path), 9)

def plot_image_with_boxes(img_path, label_path):
    img = cv2.imread(img_path)
    h, w, _ = img.shape
    label_file = label_path.replace('.jpg', '.txt').replace('.png', '.txt')
    if os.path.exists(label_file):
        with open(label_file, "r") as f:
            for line in f.readlines():
                cls, x_center, y_center, bw, bh = map(float, line.strip().split())
                x1 = int((x_center - bw/2) * w)
                y1 = int((y_center - bh/2) * h)
                x2 = int((x_center + bw/2) * w)
                y2 = int((y_center + bh/2) * h)
                color = (0, 255, 0) if cls == 0 else (0, 0, 255)
                cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
                cls_name = "smoke" if cls == 0 else "fire"
                cv2.putText(img, cls_name, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img

plt.figure(figsize=(18, 12))
for i, img_name in enumerate(sample_imgs):
    img_path = os.path.join(train_img_path, img_name)
    label_path = os.path.join(train_label_path, img_name)
    plt.subplot(3, 3, i+1)
    plt.imshow(plot_image_with_boxes(img_path, label_path))
    plt.axis('off')
    plt.title(img_name)
plt.tight_layout()
plt.show()

In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO

In [ ]:
model = YOLO("yolo12n.pt")

In [ ]:
results = model.train(data="/kaggle/working/data.yaml",epochs=60,imgsz=640,verbose=True)

In [ ]:
model.val()

In [ ]:
import shutil, os

# The trained model is automatically saved here by YOLO
best_pt = "/kaggle/working/runs/detect/train/weights/best.pt"

# Copy to output root so it's easy to download
os.makedirs("/kaggle/working/final_model", exist_ok=True)
shutil.copy2(best_pt, "/kaggle/working/final_model/fire_smoke_best.pt")
print("✅ best.pt saved to /kaggle/working/final_model/")

In [ ]:
from ultralytics import YOLO

# Load the trained best model
trained_model = YOLO("/kaggle/working/final_model/fire_smoke_best.pt")

# Export to ONNX format
trained_model.export(format="onnx", imgsz=640, opset=12)
print("✅ ONNX model exported!")

# Copy the ONNX file to final_model folder
onnx_path = "/kaggle/working/final_model/fire_smoke_best.onnx"
shutil.copy2("/kaggle/working/runs/detect/train/weights/best.onnx", onnx_path)
print(f"✅ ONNX saved to: {onnx_path}")

In [ ]:
from IPython.display import Image, display

# Show training loss/mAP curves
results_img = "/kaggle/working/runs/detect/train/results.png"
if os.path.exists(results_img):
    display(Image(results_img))
    print("📊 Training results displayed above")

# Show confusion matrix
conf_matrix = "/kaggle/working/runs/detect/train/confusion_matrix.png"
if os.path.exists(conf_matrix):
    display(Image(conf_matrix))